# AFIB‑LITE Unified Pipeline Notebook

This notebook demonstrates the **full end‑to‑end workflow** for the AFIB‑LITE project. It loads any ECG dataset placed under `model/dataset/`, uses the **adapter** module to normalise the data, runs the integer processing chain (`rantai.py`), and produces a **triage** output (Normal / AFIB / Undecided).

Structure:
- **Configuration** – path to dataset, selection of adapter.
- **Adapter loading** – any of the supported formats (WFDB, CSV, NPZ, ATtys binary).
- **Processing pipeline** – call `rantai.rantai_process`.
- **Triage** – simple rule‑based decision based on extracted features.

Add new adapters in `model/rantai-afib-lite/tools/adapter.py` and they become instantly usable here.

In [1]:
# ----- Configuration -----
import os, json

# Root folder of all ECG records
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), 'dataset'))

# Choose adapter class name (must exist in adapter.py)
# Options: 'DatasetBulk', 'DatasetStream', 'AttysBulk', 'AttysStream'
ADAPTER_CLASS = 'DatasetBulk'

# Output folder for triage results
OUT_DIR = os.path.abspath('output')
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
# ----- Helper: load adapter -----
import os, importlib.util
adapter_path = os.path.abspath(os.path.join(os.getcwd(), 'rantai-afib-lite', 'tools', 'adapter.py'))
spec = importlib.util.spec_from_file_location('adapter', adapter_path)
adapter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(adapter)
DatasetBulk = adapter.DatasetBulk
DatasetStream = adapter.DatasetStream
AttysBulk = adapter.AttysBulk
AttysStream = adapter.AttysStream
ADAPTER_MAP = {
    'DatasetBulk': DatasetBulk,
    'DatasetStream': DatasetStream,
    'AttysBulk': AttysBulk,
    'AttysStream': AttysStream,
}
AdapterCls = ADAPTER_MAP.get(ADAPTER_CLASS)
if AdapterCls is None:
    raise ValueError(f'Unknown adapter {ADAPTER_CLASS}')

In [3]:
# ----- Load all records in DATA_ROOT -----
records = []
for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(('.dat', '.hea', '.csv', '.npz', '.bin')):
            records.append(os.path.join(root, f))

print(f'Found {len(records)} ECG files')

Found 0 ECG files


In [4]:
# ----- Process each record -----
import os, importlib.util
rantai_path = os.path.abspath(os.path.join(os.getcwd(), 'rantai-afib-lite', 'tools', 'rantai.py'))
spec = importlib.util.spec_from_file_location('rantai', rantai_path)
rantai = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rantai)
rantai_process = rantai.rantai_process

triage_results = []
for rec_path in records:
    adapter = AdapterCls(rec_path)
    signal, fs = adapter.signal, adapter.fs
    proc = rantai_process(signal, fs)
    ratio = proc.get('ratio')
    entropy = proc.get('entropy')
    if ratio is None or entropy is None:
        label = 'Undecided'
    elif ratio > 0.8 and entropy < 1.2:
        label = 'AFIB'
    elif ratio < 0.5:
        label = 'Normal'
    else:
        label = 'Undecided'
    triage_results.append({
        'record': os.path.basename(rec_path),
        'label': label,
        'ratio': ratio,
        'entropy': entropy
    })
    out_path = os.path.join(OUT_DIR, os.path.basename(rec_path) + '.json')
    with open(out_path, 'w') as fp:
        json.dump(triage_results[-1], fp, indent=2)

print('Triage completed')

Triage completed


In [5]:
# ----- Summary table (pandas) -----
import pandas as pd
df = pd.DataFrame(triage_results)
display(df.head())

summary_path = os.path.join(OUT_DIR, 'triage_summary.csv')
df.to_csv(summary_path, index=False)
print('Summary saved to', summary_path)

""


Summary saved to c:\Users\Sam\Personal Data\Project\aviv-branch\nabdha-afib-lite\model\output\triage_summary.csv


## Extending the notebook
- **Add new dataset format**: implement a new class in `tools/adapter.py` with `signal` and `fs` attributes. No notebook changes required – just set `ADAPTER_CLASS` to the new name.
- **Change triage logic**: edit the rule block in the *Process each record* cell.
- **Visualise a record**: import `matplotlib.pyplot as plt` and plot `signal[:5*fs]` before processing.

The notebook is deliberately minimal; all heavy lifting stays in the reusable modules under `model/rantai-afib-lite/tools`.